In [30]:
import datasets
import json
from tqdm import tqdm
from functools import partial

In [31]:
seed = 2024
import torch
from transformers import AutoTokenizer

# model_id = "/data/models/gemma-2b"
# model_id = "/data/models/llama-2-7b"
model_id = "/data/models/llama-3-8b"
# model_id = "/data/models/Mistral-7B-v0.1"
truncation = True
model_max_length = 2048
tokenizer = AutoTokenizer.from_pretrained(model_id, model_max_length=model_max_length, use_fast=True, truncation=truncation)
tokenizer.pad_token = tokenizer.unk_token
from conversation import get_conv_template
# conv = get_conv_template("gemma")
conv = get_conv_template("llama-3")
# conv = get_conv_template("vicuna_v1.1")

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


In [32]:
from datasets import Dataset
zip_data_path = '/zhdd/home/mjyin/Projects/LLaMA-Factory/data/kto/kto_zip_10000.json'
random_data_path = '/zhdd/home/mjyin/Projects/LLaMA-Factory/data/token_kto_llama3/token_kto_random.json'
score_data_path = '/zhdd/home/mjyin/Projects/LLaMA-Factory/data/token_kto_llama3/token_kto_score.json'
raw_dataset = json.load(open(score_data_path))
raw_dataset = Dataset.from_list(raw_dataset)

In [33]:
def truncate_long_seq(sample):
    new_sample = {}
    roles = {"human": conv.roles[0], "gpt": conv.roles[1]}
    conv.messages = []
    for j, sentence in enumerate(sample['messages']):
        role = roles[sentence["from"]]
        assert role == conv.roles[j % 2]
        conv.append_message(role, sentence["value"])
    conversation = conv.get_prompt()
    input_ids = tokenizer(conversation)['input_ids']
    new_sample = {
        'messages': input_ids,
        'token_num': len(input_ids),
    }
    return new_sample
truncate_long = partial(
    truncate_long_seq,
)
tokenized_dataset = raw_dataset.map(
    truncate_long,
    num_proc=64,
    desc="Truncating Long Sequences",
)

Truncating Long Sequences (num_proc=64): 100%|██████████| 7710/7710 [00:06<00:00, 1268.69 examples/s]


In [34]:
debug = tokenized_dataset['token_num']
print(sum(debug) / len(debug))

489.24085603112843
